# 变分自编码器 VAE：学习生成数据
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：06_深度学习 | 源文件：变分自编码器_VAE.py | 核心功能：概率编码、重参数化技巧、ELBO 损失、样本生成

## 概述

VAE 在自编码器基础上引入概率建模。编码器不输出确定性向量，而是输出均值和方差，定义一个高斯分布。从该分布采样得到潜在向量，解码器重建。可以生成全新样本。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
# --- 导入库 ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

## 数学原理

### 1. VAE 的概率图模型

VAE 假设数据 $x$ 由潜变量 $z$ 生成，其联合分布为：

$$p_\theta(x, z) = p_\theta(x|z) p(z)$$

其中先验 $p(z) = \mathcal{N}(0, I)$。目标是最大化边际对数似然：

$$\log p_\theta(x) = \log \int p_\theta(x|z) p(z) dz$$

直接计算积分不可行（需要遍历所有 $z$），因此引入变分推断。

### 2. 证据下界（ELBO）

用近似后验 $q_\phi(z|x)$（编码器）替代真实后验 $p_\theta(z|x)$，推导出对数似然的下界：

$$\log p_\theta(x) \geq \mathcal{L}(\theta, \phi; x) = \underbrace{\mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)]}_{\text{重建项}} - \underbrace{KL(q_\phi(z|x) \| p(z))}_{\text{正则项}}$$

**推导过程**（KL 散度分解）：

$$KL(q_\phi(z|x) \| p_\theta(z|x)) = \mathbb{E}_{q_\phi}\left[\log \frac{q_\phi(z|x)}{p_\theta(z|x)}\right] = \mathbb{E}_{q_\phi}\left[\log \frac{q_\phi(z|x) p(x)}{p_\theta(x,z)}\right]$$

整理得：

$$\log p(x) = \mathcal{L}(\theta, \phi; x) + KL(q_\phi(z|x) \| p_\theta(z|x))$$

由于 KL 散度 $\geq 0$，$\mathcal{L}$ 是 $\log p(x)$ 的下界。

### 3. 重参数化技巧

ELBO 中的期望 $\mathbb{E}_{q_\phi(z|x)}$ 需要对 $z$ 采样，但采样操作不可微。**重参数化技巧**：

$$z = \mu + \sigma \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

将随机性转移到 $\epsilon$，使 $z$ 对 $\mu, \sigma$ 可微。代码中：

```python
def reparameterize(self, mu, logvar):
    std = torch.exp(0.5 * logvar)    # σ = exp(½ log σ²)
    eps = torch.randn_like(std)       # ε ~ N(0, I)
    return mu + eps * std             # z = μ + σ·ε
```

### 4. 两项损失的具体形式

**KL 散度项**（$q_\phi(z|x) = \mathcal{N}(\mu, \text{diag}(\sigma^2))$ 与 $p(z) = \mathcal{N}(0, I)$ 之间）：

$$KL = -\frac{1}{2}\sum_{j=1}^{d}\left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

代码中 `kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())` 精确对应此公式。

**重建项**（假设 $p_\theta(x|z)$ 为高斯分布）：

$$-\log p_\theta(x|z) \propto \|x - \hat{x}\|^2$$

代码中用 `MSELoss(reduction="sum")` 实现。

### 5. 总损失函数

$$\mathcal{L}_{VAE} = \underbrace{\sum_{i}(x_i - \hat{x}_i)^2}_{\text{重建损失}} \underbrace{- \frac{1}{2}\sum_{j}(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2)}_{\text{KL 正则}}$$

两项的物理意义：
- **重建损失**：解码器能从 $z$ 准确还原 $x$
- **KL 正则**：编码器输出的分布接近标准正态，保证潜在空间的连续性和可采样性

### 6. 编码器网络的数学角色

代码中编码器输出两个向量：

| 层 | 输出 | 数学含义 |
|----|------|----------|
| `fc_mu` | $\mu_\phi(x)$ | 后验分布的均值 |
| `fc_logvar` | $\log \sigma_\phi^2(x)$ | 后验分布的对数方差（数值稳定性） |

使用 $\log \sigma^2$ 而非直接输出 $\sigma$ 的原因：$\sigma > 0$ 的约束通过 `exp()` 自动满足，且梯度更稳定。

### 7. 生成新样本

训练完成后，解码器可直接从先验采样生成：

$$z_{new} \sim \mathcal{N}(0, I), \quad \hat{x} = p_\theta(x|z_{new})$$

这是 VAE 相比普通自编码器的核心优势：潜在空间是规则的高斯分布，任意点都能解码为有效样本。

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `fc_mu(h)` | $\mu_\phi(x)$：编码器输出均值 |
| `fc_logvar(h)` | $\log \sigma_\phi^2(x)$：编码器输出对数方差 |
| `reparameterize(mu, logvar)` | $z = \mu + \sigma \cdot \epsilon$ |
| `MSELoss(reduction="sum")` | $-\log p_\theta(x|z) \propto \sum(x_i - \hat{x}_i)^2$ |
| `torch.sum(1 + logvar - mu^2 - logvar.exp())` | $KL(q_\phi \| p)$ 的闭式解 |
| `latent_dim=16` | 潜在空间维度 $d=16$ |
| `Sigmoid()` 输出层 | 输出归一化到 $[0,1]$，配合 MSE 重建损失 |

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
digits = load_digits()
X = StandardScaler().fit_transform(digits.data)
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
X_train_t = torch.FloatTensor(X_train)
X_test_t = torch.FloatTensor(X_test)
# --- 继续 ---
train_loader = DataLoader(TensorDataset(X_train_t, X_train_t), batch_size=64, shuffle=True)

### 2. VAE 模型

运行 2. VAE 模型 的代码，观察算法在该环节的行为。

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, latent_dim=16):
        super().__init__()
        # 编码器: 输入 → 隐层 → 均值 + 对数方差
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden_dim), nn.ReLU())
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)       # 均值 μ
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)    # 对数方差 log(σ²)

        # 解码器: 潜在变量 → 隐层 → 重建
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid(),
# --- 继续 ---
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        """重参数化技巧：z = μ + σ × ε, ε ~ N(0,1)"""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

### 3. VAE 损失函数

运行 3. VAE 损失函数 的代码，观察算法在该环节的行为。

In [ ]:
def vae_loss(x_recon, x, mu, logvar):
    """
    VAE 损失 = 重建损失 + KL 散度
    重建损失: -E[log p(x|z)]  (用 MSE 或交叉熵近似)
    KL 散度: -0.5 × Σ(1 + log σ² - μ² - σ²)  (使潜在分布接近标准正态)
# --- 继续 ---
    """
    recon_loss = nn.MSELoss(reduction="sum")(x_recon, x)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_loss, recon_loss, kl_loss

### 4. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
model = VAE(input_dim=64, hidden_dim=128, latent_dim=16)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(f"=== VAE 模型 ===")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,}")

print("\n=== 训练 ===")
n_epochs = 100
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
# --- 赋值/配置 ---
    total_recon = 0
    total_kl = 0
    for batch_X, _ in train_loader:
        x_recon, mu, logvar = model(batch_X)
        loss, recon, kl = vae_loss(x_recon, batch_X, mu, logvar)
# --- 继续 ---
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_recon += recon.item()
# --- 继续 ---
        total_kl += kl.item()

    if (epoch + 1) % 20 == 0:
        n = len(X_train)
        print(f"  Epoch {epoch+1:>3}: Total={total_loss/n:.4f}, "
              f"Recon={total_recon/n:.4f}, KL={total_kl/n:.4f}")

### 5. 生成新样本

运行 5. 生成新样本 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 从 VAE 生成新样本 ===")
model.eval()
with torch.no_grad():
    # 从标准正态分布采样潜在向量
    z_new = torch.randn(10, 16)
    generated = model.decode(z_new).numpy()
print(f"生成 {generated.shape[0]} 个新样本，维度: {generated.shape[1]}")
print(f"生成样本的像素范围: [{generated.min():.3f}, {generated.max():.3f}]")

### 6. 潜在空间插值

运行 6. 潜在空间插值 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 潜在空间插值 ===")
with torch.no_grad():
    # 取两个真实样本的潜在表示，在中间插值
    z1, _ = model.encode(X_test_t[:1])
    z2, _ = model.encode(X_test_t[1:2])
    alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
    for alpha in alphas:
        z_interp = (1 - alpha) * z1 + alpha * z2
# --- 数值计算 ---
        x_interp = model.decode(z_interp).numpy()
        print(f"  α={alpha:.2f}: 像素均值={x_interp.mean():.3f}")

### 7. 重建质量

运行 7. 重建质量 的代码，观察算法在该环节的行为。

In [ ]:
with torch.no_grad():
    test_recon, mu, logvar = model(X_test_t)
    recon_loss = nn.MSELoss()(test_recon, X_test_t).item()
    print(f"\n=== 测试集重建 MSE: {recon_loss:.6f} ===")

print("\n=== VAE 要点 ===")
print("- 重参数化技巧: z = μ + σ × ε, 使得采样过程可微分")
print("- KL 散度: 约束潜在分布接近标准正态 N(0,I)")
print("- 生成新样本: 从 N(0,I) 采样 z → 解码器 → 新样本")
print("- 潜在空间连续: 相似的输入在潜在空间中也相近")
# --- 输出结果 ---
print("- 应用: 图像生成、异常检测、数据增强、表示学习")

## 关键代码解释

### 重参数化技巧

```python
z = mu + sigma * epsilon  # epsilon ~ N(0,1)
```

直接从高斯分布采样不可微。重参数化技巧将随机性转移到 epsilon，使梯度可以反向传播。

### ELBO 损失

```python
loss = reconstruction_loss + KL_divergence
```

重建损失让输出接近输入，KL 散度让潜在分布接近标准正态。

## 注意事项

1. **KL 散度的权重**：太大会导致"后验坍塌"（忽略输入）
2. **生成质量不如 GAN**：但训练更稳定
3. **潜在空间有结构**：可以插值生成过渡样本

## 总结与延伸

以上代码展示了 **变分自编码器 VAE** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **条件 VAE**：给定标签生成特定类别
- **VQ-VAE**：离散潜在空间，Stable Diffusion 的基础
- **扩散模型**：比 VAE 更强大的生成模型
